# DeepWeeds — Nguyen Tuan Thanh — 2A202602640


In [ ]:
from pathlib import Path
import os, sys, torch
BASE = Path.home() / 'research777'
ROOT = BASE / 'code'
CODE = ROOT / 'submissions/2A202602640_nguyen_tuan_thanh/code'
DATA = BASE / 'data'
OUTPUT = BASE / 'output/rtx6000_epochs12_v1/2A202602640_nguyen_tuan_thanh'
NOTEBOOK_LINK = ''
assert torch.cuda.is_available(), 'Python chưa nhận GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('Kết quả:', OUTPUT)
os.environ['LAB_REPO_DIR'] = str(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(CODE))
import colab_runner as runner
plan = runner.make_plan('deadline', 12, 64, workers=2)


In [ ]:
# Các kiểm tra CPU trước training (không tốn GPU để train model).
import subprocess
subprocess.run([sys.executable, str(CODE / 'selftest.py')], check=True)
subprocess.run([sys.executable, str(CODE / 'test_workflow.py')], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(ROOT / 'tests')], cwd=ROOT, check=True)


In [ ]:
# Tải ~490 MB, kiểm tra checksum, giải nén trên /content cho nhanh.
runner.prepare(DATA)
lab = runner.create_lab(DATA, OUTPUT, plan)
runner.run_stages(lab, ['0'])  # kiểm tra split/EDA/overfit một batch


## Vòng 1 — So sánh 5 backbone

Giữ cùng seed, split và công thức. Ô này in thời gian thật để ước lượng phần còn lại.
Các lần đã hoàn thành được bỏ qua khi resume, sau khi kiểm tra checkpoint và cấu hình.


In [ ]:
runner.run_stages(lab, ['1'])


## Vòng 2 — Công thức huấn luyện và 3 seed

Khởi tạo: fine-tune/đóng băng. Augmentation: cơ bản/CutMix. Loss: CE/label smoothing.
Thử kết hợp tốt nhất; tái sử dụng cấu hình trùng chính xác, có ghi nguồn trong log.


In [ ]:
runner.run_stages(lab, ['2'])


## Vòng 3 — Suy luận trên validation

TTA, crop, độ phân giải, gộp logit/xác suất, ensemble, temperature scaling và FP16/AMP.
Độ trễ warmup 10, đo 50 lần, có đồng bộ GPU, batch 1 và batch 32.


In [ ]:
runner.run_stages(lab, ['3'])


## Chung kết — Mở test sau khi chốt cấu hình

Chạy test một lần mỗi seed; marker và cache ngăn chạy lại khi phiên bị ngắt.
Các file chưa/sau hiệu chuẩn được tạo từ cùng lượt forward.


In [ ]:
runner.run_stages(lab, ['4'])


## Xuất bài nộp

Điền link notebook của bạn vào `NOTEBOOK_LINK`, sau đó chạy ô dưới.
Đọc lại `report.md`, xác nhận nhận xét ảnh sai đã đánh dấu và đặt quyền chia sẻ notebook.


In [ ]:
runner.run_stages(lab, ['5'], notebook_link=NOTEBOOK_LINK or None)
archive = OUTPUT.parent / 'submission_2A202602640.zip'
print("Bài nộp lưu trên Drive:", archive)
from IPython.display import Markdown, display
display(Markdown((OUTPUT / 'report.md').read_text()))
